In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE


# 0) Config

In [ ]:
DATA_PATH = "/content/bank-additional-full.csv"
OUTPUT_DIR = "/content/marketing_preprocessing_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET = "y"
SEED = 42

TEST_SIZE = 0.20
VAL_SIZE_WITHIN_DEV = 0.25   # 0.25 of 80% = 20%

CORR_THRESHOLD = 0.90
RARE_CATEGORY_MIN_SHARE = 0.01

# 1) Load data

In [ ]:
df = pd.read_csv(DATA_PATH, sep=";")

print("Initial shape:", df.shape)
display(df.head())

Initial shape: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [ ]:
df["y"].isna().sum()

np.int64(0)

# 2) Target & drop duration & duplicates

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

# remove rows with missing target values
df = df.dropna(subset=[TARGET])

# target encoding
df[TARGET] = df[TARGET].map({"no": 0, "yes": 1}).astype(int)

# remove leakage feature
LEAKAGE_FEATURES = ["duration"]

df = df.drop(columns=[c for c in LEAKAGE_FEATURES if c in df.columns])

print("Shape after duplicate removal and leakage drop:", df.shape)
print(df[TARGET].value_counts(normalize=True))

Shape after duplicate removal and leakage drop: (41176, 20)
y
0    0.887337
1    0.112663
Name: proportion, dtype: float64


# 3) Train / validation / test split

In [ ]:
train_raw_marketing, test_raw_marketing = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df[TARGET]
)

train_raw_marketing, val_raw_marketing = train_test_split(
    train_raw_marketing,
    test_size=VAL_SIZE_WITHIN_DEV,
    random_state=SEED,
    stratify=train_raw_marketing[TARGET]
)

print("Train shape:", train_raw_marketing.shape)
print("Validation shape:", val_raw_marketing.shape)
print("Test shape:", test_raw_marketing.shape)


Train shape: (24705, 20)
Validation shape: (8235, 20)
Test shape: (8236, 20)


In [ ]:
# save for AutoML models
train_raw_marketing.to_csv(os.path.join(OUTPUT_DIR, "train_raw_marketing.csv"), index=False)
val_raw_marketing.to_csv(os.path.join(OUTPUT_DIR, "val_raw_marketing.csv"), index=False)
test_raw_marketing.to_csv(os.path.join(OUTPUT_DIR, "test_raw_marketing.csv"), index=False)


# 4) Distributions

In [ ]:
def plot_target_distribution(data, target):
    data[target].value_counts().sort_index().plot(kind="bar")
    plt.title("Target Distribution")
    plt.xlabel("Target")
    plt.ylabel("Count")
    plt.xticks([0, 1], ["no", "yes"], rotation=0)
    plt.show()

def plot_numeric_distributions(data, numeric_cols):
    for col in numeric_cols:
        plt.figure(figsize=(7, 4))
        data[col].hist(bins=30)
        plt.title(f"Distribution: {col}")
        plt.xlabel(col)
        plt.ylabel("Frequency")
        plt.grid(True)
        plt.show()

plot_target_distribution(train_raw_marketing, TARGET)

numeric_cols_raw = train_raw_marketing.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_cols_raw = [c for c in numeric_cols_raw if c != TARGET]

plot_numeric_distributions(train_raw_marketing, numeric_cols_raw)

# 5)

In [ ]:
def add_logreg_features(data):
    data = data.copy()

    if "pdays" in data.columns:
        data["pdays_was_999"] = (data["pdays"] == 999).astype(int)

        data["pdays_clean"] = data["pdays"].replace(999, np.nan)

        data["pdays_clean_log"] = np.log1p(data["pdays_clean"])

        data = data.drop(columns=["pdays", "pdays_clean"])

    if "campaign" in data.columns:
        data["campaign_log"] = np.log1p(data["campaign"])
        data = data.drop(columns=["campaign"])

    if "previous" in data.columns:
        data["had_previous_contact"] = (data["previous"] > 0).astype(int)
        data["previous_log"] = np.log1p(data["previous"])
        data = data.drop(columns=["previous"])

    return data

train_fe = add_logreg_features(train_raw_marketing)
val_fe = add_logreg_features(val_raw_marketing)
test_fe = add_logreg_features(test_raw_marketing)

# 6) Seperate features and target

In [ ]:
X_train = train_fe.drop(columns=[TARGET])
y_train = train_fe[TARGET].copy()

X_val = val_fe.drop(columns=[TARGET])
y_val = val_fe[TARGET].copy()

X_test = test_fe.drop(columns=[TARGET])
y_test = test_fe[TARGET].copy()

# 7) Columns cat & numeric

In [ ]:
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

# 8) rare category handling



In [ ]:
unknown_summary = []

for col in df.columns:
    unknown_count = (df[col] == "unknown").sum()
    unknown_share = unknown_count / len(df)

    unknown_summary.append({
        "column": col,
        "unknown_count": unknown_count,
        "unknown_share": unknown_share
    })

unknown_summary = pd.DataFrame(unknown_summary)
unknown_summary = unknown_summary.sort_values("unknown_count", ascending=False)

display(unknown_summary)

In [ ]:
# Unknown is kept as valid category unless it is very rare

rare_category_maps = {}

for col in categorical_cols:
    freq = X_train[col].value_counts(normalize=True, dropna=False)
    rare_values = freq[freq < RARE_CATEGORY_MIN_SHARE].index.tolist()
    rare_category_maps[col] = rare_values

def apply_rare_category_mapping(data, rare_maps):
    data = data.copy()
    for col, rare_values in rare_maps.items():
        if col in data.columns:
            data[col] = data[col].where(~data[col].isin(rare_values), "rare")
    return data

X_train = apply_rare_category_mapping(X_train, rare_category_maps)
X_val = apply_rare_category_mapping(X_val, rare_category_maps)
X_test = apply_rare_category_mapping(X_test, rare_category_maps)

print("Rare category maps:", rare_category_maps)

# 8) Remove strongly correlated numerical features

In [ ]:
corr_matrix = X_train[numeric_cols].corr().abs()
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
corr_drop_cols = [col for col in upper_triangle.columns if any(upper_triangle[col] > CORR_THRESHOLD)]

print("Strongly correlated numeric features dropped:", corr_drop_cols)

# delete of dataframes
X_train = X_train.drop(columns=corr_drop_cols)
X_val = X_val.drop(columns=corr_drop_cols)
X_test = X_test.drop(columns=corr_drop_cols)

# new numeric cols
numeric_cols = [c for c in numeric_cols if c not in corr_drop_cols]

#9) Windowing

In [ ]:
for col in numeric_cols:
    q1 = X_train[col].quantile(0.01)
    q99 = X_train[col].quantile(0.99)

    # clipping X_train, X_val and X_test seperatly (Data Leakage)
    X_train[col] = X_train[col].clip(q1, q99)
    X_val[col] = X_val[col].clip(q1, q99)
    X_test[col] = X_test[col].clip(q1, q99)

# 9) Preprocessing: Numeric: median imputation + scaling | Categorical: one-hot encoding

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),  # dataset has no missings, but pdays_clean creates NaNs
    ("scaler", StandardScaler())
])

# no imputer for categorical variables -- > missings encoded as "unknown" and treated a sseperate categorie
categorical_pipeline = Pipeline(steps=[
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_cols),
        ("categorical", categorical_pipeline, categorical_cols)
    ],
    remainder="drop"
)

# fit only on train
X_train_processed = preprocessor.fit_transform(X_train)

# transform validation and test
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# 10)  Build DataFrames

In [ ]:
feature_names = preprocessor.get_feature_names_out()

train_preprocessed_marketing = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

val_preprocessed_marketing = pd.DataFrame(
    X_val_processed,
    columns=feature_names,
    index=X_val.index
)

test_preprocessed_marketing = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

train_preprocessed_marketing[TARGET] = y_train.values
val_preprocessed_marketing[TARGET] = y_val.values
test_preprocessed_marketing[TARGET] = y_test.values

print("Processed train shape:", train_preprocessed_marketing.shape)
print("Processed validation shape:", val_preprocessed_marketing.shape)
print("Processed test shape:", test_preprocessed_marketing.shape)

# 11) SMOTE (due to imbalanced dataset)

In [ ]:
X_train_preprocessed = train_preprocessed_marketing.drop(columns=[TARGET])
y_train_preprocessed = train_preprocessed_marketing[TARGET].astype(int)

smote = SMOTE(
    sampling_strategy="auto",
    random_state=SEED,
    k_neighbors=5
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_preprocessed,
    y_train_preprocessed
)

train_preprocessed_marketing_smote = pd.DataFrame(
    X_train_smote,
    columns=X_train_preprocessed.columns
)
train_preprocessed_marketing_smote = train_preprocessed_marketing_smote.reset_index(drop=True)

train_preprocessed_marketing_smote[TARGET] = y_train_smote.astype(int).values # to make sure the target is a integer

print("Before SMOTE:")
print(y_train_preprocessed.value_counts())

print("After SMOTE:")
print(train_preprocessed_marketing_smote[TARGET].value_counts())


In [ ]:
print("Train SMOTE shape:", train_preprocessed_marketing_smote.shape)
print("Validation shape:", val_preprocessed_marketing.shape)
print("Test shape:", test_preprocessed_marketing.shape)

# 11) Save output

In [ ]:
# excel
train_preprocessed_marketing_smote.to_excel(
    os.path.join(OUTPUT_DIR, "train_preprocessed_marketing_logreg_smote.xlsx"),
    index=False
)

val_preprocessed_marketing.to_excel(
    os.path.join(OUTPUT_DIR, "val_preprocessed_marketing_logreg.xlsx"),
    index=False
)

test_preprocessed_marketing.to_excel(
    os.path.join(OUTPUT_DIR, "test_preprocessed_marketing_logreg.xlsx"),
    index=False
)

# csv
train_preprocessed_marketing_smote.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "train_preprocessed_marketing_logreg_smote.csv"
    ),
    index=False
)

val_preprocessed_marketing.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "val_preprocessed_marketing_logreg.csv"
    ),
    index=False
)

test_preprocessed_marketing.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "test_preprocessed_marketing_logreg.csv"
    ),
    index=False
)